In [ ]:
import pandas as _hex_pandas
import datetime as _hex_datetime
import json as _hex_json

In [ ]:
hex_scheduled = _hex_json.loads("false")

In [ ]:
hex_user_email = _hex_json.loads("\"example-user@example.com\"")

In [ ]:
hex_user_attributes = _hex_json.loads("{}")

In [ ]:
hex_run_context = _hex_json.loads("\"logic\"")

In [ ]:
hex_timezone = _hex_json.loads("\"UTC\"")

In [ ]:
hex_project_id = _hex_json.loads("\"019cd7cc-aa9d-7004-acde-a1214e1396d6\"")

In [ ]:
hex_project_name = _hex_json.loads("\"Sehetna Forecasting Model\"")

In [ ]:
hex_status = _hex_json.loads("\"\"")

In [ ]:
hex_categories = _hex_json.loads("[]")

In [ ]:
hex_color_palette = _hex_json.loads("[\"#4C78A8\",\"#F58518\",\"#E45756\",\"#72B7B2\",\"#54A24B\",\"#EECA3B\",\"#B279A2\",\"#FF9DA6\",\"#9D755D\",\"#BAB0AC\"]")

In [ ]:
!uv pip install torch accelerate pytorch-forecasting pytorch-lightning --quiet

In [ ]:
# import jinja2
# raw_query = """
#     select *
#     from "25_countries_main.csv" as data_df
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
import pandas as pd
import numpy as np

FEATURES = [
    'temp_squared', 'pm25_ugm3', 'heat_wave_days', 'month_sin',
    'temp_precip_interaction', 'aqi_pm', 'temp_anomaly_celsius',
    'pollution_vulnerability', 'flood_indicator', 'healthcare_access_index',
    'distance_to_equator', 'pm25_change_rate', 'pm25_ugm3_lag_1w', 'month_cos',
    'temp_change_rate', 'temperature_celsius', 'gdp_per_capita_usd',
    'quarter', 'spatial_lag_pm25', 'is_northern',
    'is_tropical', 'spatial_lag_temp', 'food_security_index',
]

TARGETS = [
    'respiratory_disease_rate', 'cardio_mortality_rate',
    'vector_disease_risk_score', 'waterborne_disease_incidents',
    'heat_related_admissions',
]

SEQ_LEN = 78
HORIZON_LEN = 12
NUM_FEATURES = len(FEATURES)
NUM_TARGETS = len(TARGETS)
ALL_COLS = FEATURES + TARGETS

df = data_df.copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['country_code', 'date']).reset_index(drop=True)

# --- Direct computations ---
df['temp_squared'] = df['temperature_celsius'] ** 2
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['quarter'] = (df['month'] - 1) // 3 + 1
df['temp_precip_interaction'] = df['temperature_celsius'] * df['precipitation_mm']
df['pollution_vulnerability'] = df['pm25_ugm3'] / (df['healthcare_access_index'] + 1e-6)
df['distance_to_equator'] = df['latitude'].abs()
df['is_northern'] = (df['latitude'] > 0).astype(int)
df['is_tropical'] = (df['latitude'].abs() < 23.5).astype(int)

# --- Per-country temporal features ---
df['pm25_change_rate'] = df.groupby('country_code')['pm25_ugm3'].diff()
df['temp_change_rate'] = df.groupby('country_code')['temperature_celsius'].diff()
df['pm25_ugm3_lag_1w'] = df.groupby('country_code')['pm25_ugm3'].shift(1)

# --- Spatial lag (leave-one-out mean per week) ---
for col, new_col in [('pm25_ugm3', 'spatial_lag_pm25'), ('temperature_celsius', 'spatial_lag_temp')]:
    week_sum = df.groupby('date')[col].transform('sum')
    week_count = df.groupby('date')[col].transform('count')
    df[new_col] = (week_sum - df[col]) / (week_count - 1)

# --- Backfill NaN from diff/shift (first row per country only) ---
for col in ['pm25_change_rate', 'temp_change_rate', 'pm25_ugm3_lag_1w']:
    df[col] = df.groupby('country_code')[col].bfill()

# --- Validate ---
missing_feats = set(FEATURES) - set(df.columns)
missing_tgts = set(TARGETS) - set(df.columns)
assert not missing_feats, f"Missing features: {missing_feats}"
assert not missing_tgts, f"Missing targets: {missing_tgts}"

nan_counts = df[ALL_COLS].isna().sum()
nan_cols = nan_counts[nan_counts > 0]
assert nan_cols.empty, f"NaN found:\n{nan_cols}"

df_engineered = df
print(f"Shape: {df_engineered.shape[0]} rows × {df_engineered.shape[1]} cols")
print(f"Features: {NUM_FEATURES} | Targets: {NUM_TARGETS}")
print(f"Countries: {df_engineered['country_code'].nunique()} | Date range: {df['date'].min().date()} → {df['date'].max().date()}")

Shape: 14100 rows × 46 cols
Features: 23 | Targets: 5
Countries: 25 | Date range: 2015-01-04 → 2025-10-19


In [ ]:
from sklearn.preprocessing import RobustScaler

train_frames, val_frames, test_frames = [], [], []
scalers = {}

for cc, grp in df_engineered.groupby('country_code'):
    grp = grp.sort_values('date').reset_index(drop=True)
    n = len(grp)

    train_end = int(n * 0.80)
    val_end = int(n * 0.90)

    train_part = grp.iloc[:train_end].copy()
    val_part = grp.iloc[train_end:val_end].copy()
    test_part = grp.iloc[val_end:].copy()

    # Fit scaler on train only — RobustScaler uses median/IQR, preserves peaks
    scaler = RobustScaler()
    scaler.fit(train_part[ALL_COLS])
    scalers[cc] = scaler

    train_part[ALL_COLS] = scaler.transform(train_part[ALL_COLS])
    val_part[ALL_COLS] = scaler.transform(val_part[ALL_COLS])
    test_part[ALL_COLS] = scaler.transform(test_part[ALL_COLS])

    train_frames.append(train_part)
    val_frames.append(val_part)
    test_frames.append(test_part)

train_df = pd.concat(train_frames).reset_index(drop=True)
val_df = pd.concat(val_frames).reset_index(drop=True)
test_df = pd.concat(test_frames).reset_index(drop=True)

print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows | Test: {len(test_df)} rows")
print(f"Per-country split: ~{len(train_df)//25} train, ~{len(val_df)//25} val, ~{len(test_df)//25} test weeks")
print(f"Scalers fitted for {len(scalers)} countries")

Train: 11275 rows | Val: 1400 rows | Test: 1425 rows
Per-country split: ~451 train, ~56 val, ~57 test weeks
Scalers fitted for 25 countries


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


# =============================================================================
# GENERIC SLIDING-WINDOW DATASET (reusable across model architectures)
# =============================================================================
class SlidingWindowDataset(Dataset):
    """
    Sliding-window dataset for multi-country time series.
    
    Produces (X, y) pairs where:
        X = [seq_len, num_features + num_targets]  (input context)
        y = [horizon_len, num_targets]              (forecast targets)
    """

    def __init__(self, windows: np.ndarray, labels: np.ndarray):
        self.windows = torch.tensor(windows, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx], self.labels[idx]


# =============================================================================
# WINDOW BUILDERS
# =============================================================================
def _extract_windows(values: np.ndarray, seq_len: int, horizon_len: int,
                     target_indices: list, min_label_start: int = 0):
    """Core loop: slide over a single country's value array and extract (X, y) pairs."""
    windows, labels = [], []
    n = len(values)
    for i in range(n - seq_len - horizon_len + 1):
        label_start = i + seq_len
        if label_start < min_label_start:
            continue
        x = values[i : i + seq_len]
        y = values[label_start : label_start + horizon_len][:, target_indices]
        if len(y) == horizon_len:
            windows.append(x)
            labels.append(y)
    return windows, labels


def build_windows(df, features, targets, seq_len, horizon_len):
    """Build sliding windows from a single split (training set)."""
    all_cols = features + targets
    target_indices = list(range(len(features), len(all_cols)))
    windows, labels = [], []

    for _, grp in df.groupby('country_code'):
        vals = grp.sort_values('date')[all_cols].values.astype(np.float32)
        w, l = _extract_windows(vals, seq_len, horizon_len, target_indices)
        windows.extend(w)
        labels.extend(l)

    return np.array(windows, dtype=np.float32), np.array(labels, dtype=np.float32)


def build_windows_with_context(current_df, prior_df, features, targets,
                                seq_len, horizon_len):
    """Build windows for val/test, borrowing context from the prior split."""
    all_cols = features + targets
    target_indices = list(range(len(features), len(all_cols)))
    windows, labels = [], []

    for cc in current_df['country_code'].unique():
        prior_grp = prior_df[prior_df['country_code'] == cc].sort_values('date')
        curr_grp = current_df[current_df['country_code'] == cc].sort_values('date')

        context = prior_grp.tail(seq_len)
        combined = pd.concat([context, curr_grp]).reset_index(drop=True)
        vals = combined[all_cols].values.astype(np.float32)
        context_len = len(context)

        w, l = _extract_windows(vals, seq_len, horizon_len, target_indices,
                                min_label_start=context_len)
        windows.extend(w)
        labels.extend(l)

    return np.array(windows, dtype=np.float32), np.array(labels, dtype=np.float32)


def create_dataloaders(train_df, val_df, test_df, features, targets,
                       seq_len, horizon_len, batch_size=128):
    """One-call factory: build windows → datasets → dataloaders."""
    train_w, train_l = build_windows(train_df, features, targets, seq_len, horizon_len)

    val_w, val_l = build_windows_with_context(
        val_df, train_df, features, targets, seq_len, horizon_len)

    test_w, test_l = build_windows_with_context(
        test_df, pd.concat([train_df, val_df]), features, targets, seq_len, horizon_len)

    train_ds = SlidingWindowDataset(train_w, train_l)
    val_ds   = SlidingWindowDataset(val_w, val_l)
    test_ds  = SlidingWindowDataset(test_w, test_l)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

    return {
        'train': {'dataset': train_ds, 'loader': train_loader, 'windows': train_w, 'labels': train_l},
        'val':   {'dataset': val_ds,   'loader': val_loader,   'windows': val_w,   'labels': val_l},
        'test':  {'dataset': test_ds,  'loader': test_loader,  'windows': test_w,  'labels': test_l},
    }


# --- Build all dataloaders ---
BATCH_SIZE = 128
data = create_dataloaders(
    train_df, val_df, test_df,
    FEATURES, TARGETS,
    SEQ_LEN, HORIZON_LEN,
    batch_size=BATCH_SIZE,
)

train_dataset, train_loader = data['train']['dataset'], data['train']['loader']
val_dataset,   val_loader   = data['val']['dataset'],   data['val']['loader']
test_dataset,  test_loader  = data['test']['dataset'],  data['test']['loader']
val_windows,   val_labels   = data['val']['windows'],   data['val']['labels']
test_windows,  test_labels  = data['test']['windows'],  data['test']['labels']

print(f"Train windows: {len(train_dataset):,} | Val windows: {len(val_dataset):,} | Test windows: {len(test_dataset):,}")
print(f"Window shape:  X=[{SEQ_LEN}, {NUM_FEATURES + NUM_TARGETS}] → y=[{HORIZON_LEN}, {NUM_TARGETS}]")
print(f"Batches/epoch: {len(train_loader)}")


Train windows: 9,050 | Val windows: 1,125 | Test windows: 1,150
Window shape:  X=[78, 28] → y=[12, 5]
Batches/epoch: 71


In [ ]:
import torch.nn as nn


# =============================================================================
# TEMPORAL FUSION TRANSFORMER — adapted for sliding-window (X, y) input
# =============================================================================
class GatedResidualNetwork(nn.Module):
    """GRN: core building block of TFT for nonlinear processing with skip connections."""

    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.1, context_dim=None):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()
        self.layer_norm = nn.LayerNorm(output_dim)
        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()
        self.context_proj = nn.Linear(context_dim, hidden_dim, bias=False) if context_dim else None

    def forward(self, x, context=None):
        residual = self.skip(x)
        h = self.fc1(x)
        if self.context_proj is not None and context is not None:
            h = h + self.context_proj(context)
        h = self.elu(h)
        h = self.dropout(h)
        gate_val = self.sigmoid(self.gate(h))
        h = self.fc2(h)
        out = self.layer_norm(gate_val * h + residual)
        return out


class VariableSelectionNetwork(nn.Module):
    """VSN: learns which input variables matter most at each time step."""

    def __init__(self, num_vars, d_model, dropout=0.1, context_dim=None):
        super().__init__()
        self.num_vars = num_vars
        self.d_model = d_model
        self.var_grns = nn.ModuleList([
            GatedResidualNetwork(1, d_model, d_model, dropout) for _ in range(num_vars)
        ])
        self.softmax_grn = GatedResidualNetwork(
            num_vars * d_model, d_model, num_vars, dropout, context_dim=context_dim
        )

    def forward(self, x, context=None):
        # x: [B, T, num_vars]
        var_outputs = []
        for i in range(self.num_vars):
            var_outputs.append(self.var_grns[i](x[..., i:i+1]))  # [B, T, d_model]
        var_outputs = torch.stack(var_outputs, dim=-2)  # [B, T, num_vars, d_model]

        flat = var_outputs.reshape(*x.shape[:2], -1)  # [B, T, num_vars * d_model]
        ctx = context.unsqueeze(1).expand(-1, x.size(1), -1) if context is not None else None
        weights = torch.softmax(self.softmax_grn(flat, ctx), dim=-1)  # [B, T, num_vars]

        combined = (var_outputs * weights.unsqueeze(-1)).sum(dim=-2)  # [B, T, d_model]
        return combined, weights


class TemporalFusionTransformer(nn.Module):
    """
    TFT adapted for pre-windowed (X, y) pairs.
    
    Input:  X of shape [batch, seq_len, num_input_channels]
    Output: predictions of shape [batch, horizon_len, num_targets]
    """

    def __init__(self, num_inputs, num_targets, seq_len, horizon_len,
                 d_model=64, num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.num_targets = num_targets
        self.horizon_len = horizon_len
        self.seq_len = seq_len
        self.d_model = d_model

        # Variable Selection
        self.vsn = VariableSelectionNetwork(num_inputs, d_model, dropout)

        # Temporal processing: LSTM encoder-decoder
        self.encoder_lstm = nn.LSTM(d_model, d_model, num_layers=num_layers,
                                     dropout=dropout if num_layers > 1 else 0,
                                     batch_first=True)
        self.decoder_lstm = nn.LSTM(d_model, d_model, num_layers=num_layers,
                                     dropout=dropout if num_layers > 1 else 0,
                                     batch_first=True)

        # Gated skip connection over LSTM
        self.post_lstm_gate = nn.Linear(d_model, d_model)
        self.post_lstm_norm = nn.LayerNorm(d_model)

        # Interpretable multi-head attention
        self.self_attn = nn.MultiheadAttention(d_model, num_heads,
                                                dropout=dropout, batch_first=True)
        self.post_attn_gate = nn.Linear(d_model, d_model)
        self.post_attn_norm = nn.LayerNorm(d_model)

        # Position-wise feedforward
        self.ff_grn = GatedResidualNetwork(d_model, d_model * 2, d_model, dropout)

        # Decoder input projection (learned query for future steps)
        self.future_proj = nn.Linear(d_model, d_model)
        self.future_pos = nn.Parameter(torch.randn(1, horizon_len, d_model) * 0.02)

        # Output
        self.output_proj = nn.Linear(d_model, num_targets)

    def forward(self, past_values):
        B = past_values.size(0)

        # Variable selection
        selected, var_weights = self.vsn(past_values)  # [B, seq_len, d_model]

        # Encoder LSTM
        enc_out, (h_n, c_n) = self.encoder_lstm(selected)

        # Gated skip over encoder
        gated = torch.sigmoid(self.post_lstm_gate(enc_out)) * enc_out
        enc_enriched = self.post_lstm_norm(gated + selected)

        # Decoder: generate future queries
        decoder_input = self.future_pos.expand(B, -1, -1)  # [B, horizon, d_model]
        dec_out, _ = self.decoder_lstm(decoder_input, (h_n, c_n))

        # Combine encoder + decoder for attention
        full_seq = torch.cat([enc_enriched, dec_out], dim=1)  # [B, seq+horizon, d_model]

        # Self-attention (decoder attends to full sequence)
        attn_out, _ = self.self_attn(full_seq, full_seq, full_seq)
        gated_attn = torch.sigmoid(self.post_attn_gate(attn_out)) * attn_out
        attn_enriched = self.post_attn_norm(gated_attn + full_seq)

        # Take only the decoder (future) positions
        future_repr = attn_enriched[:, self.seq_len:, :]  # [B, horizon, d_model]

        # Feed-forward + output
        out = self.ff_grn(future_repr)
        predictions = self.output_proj(out)  # [B, horizon, num_targets]
        return predictions


# =============================================================================
# INSTANTIATE TFT MODEL
# =============================================================================
tft_model = TemporalFusionTransformer(
    num_inputs=NUM_FEATURES + NUM_TARGETS,
    num_targets=NUM_TARGETS,
    seq_len=SEQ_LEN,
    horizon_len=HORIZON_LEN,
    d_model=64,
    num_heads=4,
    num_layers=2,
    dropout=0.15,
)

total_params = sum(p.numel() for p in tft_model.parameters())
trainable_params = sum(p.numel() for p in tft_model.parameters() if p.requires_grad)
print(f"TFT — Total params: {total_params:,} | Trainable: {trainable_params:,}")
print(f"Input: [{SEQ_LEN}, {NUM_FEATURES + NUM_TARGETS}] → Output: [{HORIZON_LEN}, {NUM_TARGETS}]")
print(f"Architecture: VSN({NUM_FEATURES + NUM_TARGETS} vars) → LSTM(2L, d=64) → MHA(4H) → GRN → Linear")


TFT — Total params: 600,913 | Trainable: 600,913
Input: [78, 28] → Output: [12, 5]
Architecture: VSN(28 vars) → LSTM(2L, d=64) → MHA(4H) → GRN → Linear


In [ ]:
def train_tft(model, train_loader, val_loader, device,
              num_targets, num_epochs=100, patience=25, lr=2e-3):
    """Train TFT with composite loss, OneCycleLR, and early stopping."""
    model = model.to(device)

    # Global target stats for stable WeightedMSE
    all_targets = torch.cat([y for _, y in train_loader])
    flat = all_targets.reshape(-1, num_targets)
    global_median = flat.median(dim=0).values.to(device)
    q75 = flat.quantile(0.75, dim=0).to(device)
    q25 = flat.quantile(0.25, dim=0).to(device)
    global_iqr = (q75 - q25).clamp(min=1e-6)

    criterion = CompositeLoss(
        alpha=0.3, beta=0.5, gamma=0.2,
        huber_delta=2.0, wmse_lambda=1.5,
        global_median=global_median, global_iqr=global_iqr,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-3)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=num_epochs,
        steps_per_epoch=len(train_loader), pct_start=0.3,
    )

    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': [], 'val_r2': []}

    for epoch in range(num_epochs):
        # --- Train ---
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item() * bx.size(0)
        train_loss /= len(train_loader.dataset)

        # --- Validate ---
        model.eval()
        val_loss = 0.0
        val_preds, val_tgts = [], []
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                pred = model(bx)
                loss = criterion(pred, by)
                val_loss += loss.item() * bx.size(0)
                val_preds.append(pred.cpu())
                val_tgts.append(by.cpu())
        val_loss /= len(val_loader.dataset)

        vp = torch.cat(val_preds).reshape(-1, num_targets)
        vt = torch.cat(val_tgts).reshape(-1, num_targets)
        val_mae = (vp - vt).abs().mean().item()
        val_rmse = ((vp - vt) ** 2).mean().sqrt().item()
        ss_res = ((vp - vt) ** 2).sum().item()
        ss_tot = ((vt - vt.mean(dim=0, keepdim=True)) ** 2).sum().item()
        val_r2 = 1 - ss_res / max(ss_tot, 1e-8)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae)
        history['val_rmse'].append(val_rmse)
        history['val_r2'].append(val_r2)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} "
                  f"| MAE: {val_mae:.4f} | RMSE: {val_rmse:.4f} | R²: {val_r2:.4f}")

        # --- Early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
    return model, history


# =============================================
# >>> UNCOMMENT BELOW TO START TFT TRAINING <<<
# >>> (Long-running: ~10-15 min on GPU)     <<<
# =============================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
tft_model, tft_history = train_tft(
    tft_model, train_loader, val_loader, device,
    num_targets=NUM_TARGETS, num_epochs=100, patience=25, lr=2e-3,
)


Device: cpu
Epoch   1 | Train: 1.6628 | Val: 2.0084 | MAE: 0.6353 | RMSE: 0.8513 | R²: 0.0200
Epoch   5 | Train: 1.3361 | Val: 1.7465 | MAE: 0.5990 | RMSE: 0.8174 | R²: 0.0965


In [ ]:
def mc_dropout_inference(model, data_loader, device, n_forward=50):
    """Run N stochastic forward passes with dropout active for uncertainty estimation."""
    model.to(device)
    model.train()  # keep dropout active

    all_means, all_stds, all_targets = [], [], []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            preds = torch.stack([model(batch_x).cpu() for _ in range(n_forward)])
            all_means.append(preds.mean(dim=0))
            all_stds.append(preds.std(dim=0))
            all_targets.append(batch_y)

    return torch.cat(all_means), torch.cat(all_stds), torch.cat(all_targets)


def collapse_overlapping_predictions(all_preds, all_stds, all_targets):
    """
    Collapse overlapping sliding-window predictions into final horizon.
    Used for VISUALIZATION only — not for primary evaluation.
    """
    num_seq, horizon_len, num_targets = all_preds.shape
    total_t = num_seq + horizon_len - 1

    ts_preds = {i: [] for i in range(total_t)}
    ts_vars = {i: [] for i in range(total_t)}
    ts_tgts = {i: [] for i in range(total_t)}

    for s in range(num_seq):
        for h in range(horizon_len):
            t = s + h
            ts_preds[t].append(all_preds[s, h])
            ts_vars[t].append(all_stds[s, h] ** 2)
            ts_tgts[t].append(all_targets[s, h])

    final_preds, final_stds, final_tgts = [], [], []
    for t in range(total_t - horizon_len, total_t):
        p = torch.stack(ts_preds[t])
        v = torch.stack(ts_vars[t])
        g = torch.stack(ts_tgts[t])
        final_preds.append(p.mean(dim=0))
        final_stds.append(v.mean(dim=0).sqrt())
        final_tgts.append(g.mean(dim=0))

    return torch.stack(final_preds), torch.stack(final_stds), torch.stack(final_tgts)


# =============================================
# >>> UNCOMMENT BELOW TO RUN INFERENCE <<<
# >>> (Long-running: ~45 sec on GPU)     <<<
# =============================================
# test_means, test_stds, test_targets = mc_dropout_inference(model, test_loader, device, n_forward=50)
#
# # --- PRIMARY EVALUATION: raw window-level metrics (before collapsing) ---
# from sklearn.metrics import r2_score as sk_r2
# flat_preds = test_means.reshape(-1, test_means.shape[-1]).numpy()
# flat_targets = test_targets.reshape(-1, test_targets.shape[-1]).numpy()
# print("=== Window-Level Metrics (Primary) ===")
# for i, tgt in enumerate(TARGETS):
#     mae = np.abs(flat_preds[:, i] - flat_targets[:, i]).mean()
#     rmse = np.sqrt(((flat_preds[:, i] - flat_targets[:, i]) ** 2).mean())
#     r2 = sk_r2(flat_targets[:, i], flat_preds[:, i])
#     print(f"  {tgt:>35s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
#
# # --- SECONDARY: collapse for visualization ---
# collapsed_preds, collapsed_stds, collapsed_targets = collapse_overlapping_predictions(
#     test_means, test_stds, test_targets
# )
# ci_lower = collapsed_preds - 1.96 * collapsed_stds
# ci_upper = collapsed_preds + 1.96 * collapsed_stds
# print(f"\nCollapsed shape: {collapsed_preds.shape} | 95% CI width (mean): {(ci_upper - ci_lower).mean():.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def compute_metrics(preds, targets, target_names):
    """
    Compute metrics on raw window-level predictions.
    preds/targets: [num_windows, horizon_len, num_targets] or [timesteps, num_targets]
    """
    if preds.ndim == 3:
        preds_np = preds.reshape(-1, preds.shape[-1])
        targets_np = targets.reshape(-1, targets.shape[-1])
    else:
        preds_np = preds
        targets_np = targets

    preds_np = preds_np.numpy() if hasattr(preds_np, 'numpy') else preds_np
    targets_np = targets_np.numpy() if hasattr(targets_np, 'numpy') else targets_np

    results = []
    for i, name in enumerate(target_names):
        p, t = preds_np[:, i], targets_np[:, i]

        mae = mean_absolute_error(t, p)
        rmse = np.sqrt(mean_squared_error(t, p))
        r2 = r2_score(t, p)

        threshold = np.percentile(np.abs(t), 90)
        peak_mask = np.abs(t) > threshold
        peak_mae = mean_absolute_error(t[peak_mask], p[peak_mask]) if peak_mask.sum() > 0 else np.nan

        dt, dp = np.diff(t), np.diff(p)
        dir_acc = (np.sign(dt) == np.sign(dp)).mean() * 100 if len(dt) > 0 else np.nan

        results.append({
            'Target': name,
            'MAE': round(mae, 4),
            'RMSE': round(rmse, 4),
            'R²': round(r2, 4),
            'Peak MAE': round(peak_mae, 4) if not np.isnan(peak_mae) else 'N/A',
            'Dir. Acc (%)': round(dir_acc, 2) if not np.isnan(dir_acc) else 'N/A',
        })

    return pd.DataFrame(results)


def plot_forecasts_with_ci(preds, stds, targets, target_names):
    """Plot collapsed forecasts with 95% CI bands (visualization only)."""
    n = len(target_names)
    fig, axes = plt.subplots(n, 1, figsize=(12, 3.5 * n), sharex=True)
    if n == 1:
        axes = [axes]

    t_axis = np.arange(preds.shape[0])
    for i, (ax, name) in enumerate(zip(axes, target_names)):
        p = preds[:, i].numpy() if hasattr(preds, 'numpy') else preds[:, i]
        s = stds[:, i].numpy() if hasattr(stds, 'numpy') else stds[:, i]
        a = targets[:, i].numpy() if hasattr(targets, 'numpy') else targets[:, i]

        ax.plot(t_axis, a, 'k-', lw=1.5, label='Actual')
        ax.plot(t_axis, p, 'b-', lw=1.2, label='Predicted')
        ax.fill_between(t_axis, p - 1.96 * s, p + 1.96 * s,
                         alpha=0.25, color='steelblue', label='95% CI')
        ax.set_ylabel(name, fontsize=9)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Forecast Horizon Step')
    fig.suptitle('PatchTST Forecasts with 95% Confidence Intervals', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_training_history(history):
    """Plot loss curves, validation metrics, and R²."""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

    ax1.plot(history['train_loss'], label='Train Loss', lw=1.2)
    ax1.plot(history['val_loss'], label='Val Loss', lw=1.2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Composite Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history['val_mae'], label='Val MAE', lw=1.2)
    ax2.plot(history['val_rmse'], label='Val RMSE', lw=1.2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Metric')
    ax2.set_title('Validation MAE / RMSE')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3.plot(history['val_r2'], label='Val R²', lw=1.2, color='green')
    ax3.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Baseline (mean)')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('R²')
    ax3.set_title('Validation R²')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# =============================================
# >>> UNCOMMENT BELOW TO RUN EVALUATION  <<<
# >>> (Fast: < 5 sec)                    <<<
# =============================================
# print("=== Window-Level Metrics (test set) ===")
# metrics_df = compute_metrics(test_means, test_targets, TARGETS)
# print(metrics_df.to_string(index=False))
#
# print("\n=== Training Curves ===")
# plot_training_history(history)
#
# print("\n=== Collapsed Forecast Visualization ===")
# plot_forecasts_with_ci(collapsed_preds, collapsed_stds, collapsed_targets, TARGETS)